# Week 7 Exercise – Fine-Tune Llama 3.2 3B with QLoRA

**Builds on Week 6:** In Week 6 we predicted product prices using `ed-donner/items_lite` with constant baseline and LLM (GPT-4o-mini). Here we **fine-tune** Llama 3.2 3B with QLoRA on the same task, then evaluate.

- **Base model**: Llama 3.2 3B (4-bit quantized)
- **Training**: QLoRA (LoRA on attention layers), SFTTrainer
- **Data**: `ed-donner/items_prompts_lite` (prompt/completion from items_lite)
- **Evaluation**: `util.evaluate()` – MAE, R², scatter and error-trend charts

**Run in Colab (GPU)** – make sure to add HF_TOKEN.

### Environment setup

Run the cell below to install dependencies and download `util.py` from the course repo.

In [ ]:
!pip install -q --upgrade bitsandbytes trl transformers accelerate datasets peft
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

### Imports

Libraries for model loading, quantization, PEFT, datasets, and the evaluation helper.

In [ ]:
import os
import re
import math
from datetime import datetime
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from util import evaluate

### Configuration

Base model, dataset, QLoRA and training hyperparameters. Set `HF_USER` to your HuggingFace username to push adapters to the Hub.

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
HF_USER = "ngahunj"  # Your HF username for pushing adapters
PROJECT_NAME = "price"
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"
DATASET_NAME = "ed-donner/items_prompts_lite"
QUANT_4_BIT = True

# Training hyperparameters (lite: fast run on T4)
EPOCHS = 1
BATCH_SIZE = 16
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
SAVE_STEPS = 100
LOG_STEPS = 10

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

### Hugging Face login

Add `HF_TOKEN` to Colab secrets (key icon in sidebar).

In [ ]:
try:
    hf_token = userdata.get("HF_TOKEN")
    login(hf_token, add_to_git_credential=True)

except Exception as er:
    print(er, "ERRr")

### Load and format dataset

Load `items_prompts_lite` and add a `text` column (prompt + completion) for SFTTrainer.

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset["train"]
val = dataset["val"]
test = dataset["test"]

def add_text(example):
    return {"text": example["prompt"] + example["completion"]}

train = train.map(add_text, remove_columns=train.column_names)
val = val.map(add_text, remove_columns=val.column_names)

print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

### Load base model and run QLoRA training

Load Llama 3.2 3B with 4-bit quantization, configure LoRA, and fine-tune with SFTTrainer.

In [ ]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Base model memory: {base_model.get_memory_footprint() / 1e6:.1f} MB")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_args = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_config,
    args=train_args,
)

trainer.train()
print(f"Training complete. Adapters saved to {PROJECT_RUN_NAME}/")

fine_tuned_model = trainer.model

### Prediction and evaluation

Define a predictor that takes a prompt, generates up to 8 tokens, and returns the decoded text. Same interface as Week 6's evaluator.

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

set_seed(42)
evaluate(model_predict, test)